In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    make_scorer, roc_auc_score,
    average_precision_score, f1_score
)

In [9]:
df = pd.read_csv('final_df.csv')

drop_cols = [
    'permalink',        # identifier
    'homepage_url',     # identifier
    'status',           # leakage — encodes the target directly
    'funding_total_usd' # replaced by log_funding
]

redundant_cols = [
    'founded_at',       # captured by founded_year + startup_age_days
    'founded_month',    # noise — year is sufficient
    'founded_quarter',  # noise — year is sufficient
    'first_funding_at', # captured by startup_age_days
    'last_funding_at',  # captured by funding_duration_days
]

df = df.drop(columns=drop_cols + redundant_cols)
print(f"Dataset shape after dropping columns: {df.shape}")

Dataset shape after dropping columns: (48121, 34)


In [10]:
y = df['success'].copy()
X = df.drop(columns=['name', 'success']).copy()

# Extract primary category from pipe-delimited string
X['primary_category'] = (
    X['category_list']
    .str.strip('|')
    .str.split('|')
    .str[0]
    .str.strip()
    .fillna('Unknown')
)
X = X.drop(columns=['category_list'])

# Drop high-cardinality geo columns
X = X.drop(columns=['region', 'city'], errors='ignore')

print(f"X shape: {X.shape}")
print(f"\nClass distribution:\n{y.value_counts()}")
print(f"\nClass balance: {y.mean():.2%} positive")

X shape: (48121, 30)

Class distribution:
success
0    44429
1     3692
Name: count, dtype: int64

Class balance: 7.67% positive


In [11]:
round_cols = ['seed', 'venture', 'round_A', 'round_B', 'round_C',
              'round_D', 'round_E', 'round_F', 'round_G', 'round_H']

round_order = {
    'seed': 1, 'venture': 2, 'round_A': 3, 'round_B': 4, 'round_C': 5,
    'round_D': 6, 'round_E': 7, 'round_F': 8, 'round_G': 9, 'round_H': 10
}

# Highest funding round reached (ordinal signal)
X['highest_round'] = X[round_cols].apply(
    lambda row: max([round_order[c] for c in round_cols if row[c] > 0], default=0),
    axis=1
)

# Number of distinct round types received
X['round_diversity'] = (X[round_cols] > 0).sum(axis=1)

# Funding intensity features
X['funding_per_round'] = np.expm1(X['log_funding']) / (X['funding_rounds'] + 1)
X['funding_per_year']  = np.expm1(X['log_funding']) / (X['startup_age_days'] / 365 + 1)
X['rounds_per_year']   = X['funding_rounds'] / (X['funding_duration_days'] / 365 + 1)

# Drop sparse round-type columns (>95% zeros)
sparse_cols = [
    'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing',
    'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt',
    'secondary_market', 'product_crowdfunding'
]
for col in sparse_cols:
    if col in X.columns and (X[col] == 0).mean() > 0.95:
        X = X.drop(columns=[col])

print(f"X shape after feature engineering: {X.shape}")
print(f"Engineered features added: highest_round, round_diversity, "
      f"funding_per_round, funding_per_year, rounds_per_year")

X shape after feature engineering: (48121, 26)
Engineered features added: highest_round, round_diversity, funding_per_round, funding_per_year, rounds_per_year


In [12]:
cat_cols = ['primary_category', 'market', 'country_code', 'state_code']

# Fix unknown state codes
X['state_code'] = X['state_code'].replace('Unknown', np.nan).fillna('Unknown')

# Cast to category dtype (used by LightGBM natively)
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].astype('category')

# Sanity check
print(f"Final X shape: {X.shape}")
print(f"Any NaNs: {X.isnull().sum().sum()}")
print(f"Any Inf:  {np.isinf(X.select_dtypes('number')).sum().sum()}")
print(f"\nDtype summary:\n{X.dtypes.value_counts()}")

Final X shape: (48121, 26)
Any NaNs: 0
Any Inf:  63

Dtype summary:
float64     20
int64        2
category     1
category     1
category     1
category     1
Name: count, dtype: int64


In [13]:
neg, pos = y.value_counts()[0], y.value_counts()[1]
spw = neg / pos  # ~12.0

lgbm_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.05,
    reg_lambda=0.1,
    scale_pos_weight=spw,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'roc_auc':  'roc_auc',
    'pr_auc':   make_scorer(average_precision_score, response_method='predict_proba'),
    'f1_macro': 'f1_macro'
}

lgbm_results = cross_validate(
    lgbm_model, X, y,
    cv=cv, scoring=scoring,
    return_train_score=True,
    verbose=1
)

print("\n── LightGBM CV Results ──")
lgbm_scores = {}
for metric in ['roc_auc', 'pr_auc', 'f1_macro']:
    test  = lgbm_results[f'test_{metric}']
    train = lgbm_results[f'train_{metric}']
    print(f"{metric:12s}  test: {test.mean():.4f} ± {test.std():.4f}"
          f"  |  train: {train.mean():.4f} ± {train.std():.4f}")
    lgbm_scores[metric] = test.mean()



── LightGBM CV Results ──
roc_auc       test: 0.8052 ± 0.0051  |  train: 0.9698 ± 0.0006
pr_auc        test: 0.2839 ± 0.0139  |  train: 0.7197 ± 0.0051
f1_macro      test: 0.6198 ± 0.0042  |  train: 0.7408 ± 0.0014


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   10.7s finished


In [14]:
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = X.copy()
X_encoded[cat_cols] = enc.fit_transform(X[cat_cols].astype(str))

# Replace any inf values produced by feature engineering edge cases
X_encoded = X_encoded.replace([np.inf, -np.inf], np.nan)
X_encoded = X_encoded.fillna(X_encoded.median(numeric_only=True))

assert not np.isinf(X_encoded.select_dtypes('number').values).any(), "Inf values remain"
assert not X_encoded.isnull().any().any(), "NaN values remain"
print(f"Encoded feature matrix ready: {X_encoded.shape}")

Encoded feature matrix ready: (48121, 26)


In [15]:
def run_cv(model, X_data, name):
    roc_scores, pr_scores, f1_scores = [], [], []
    for tr_idx, val_idx in cv.split(X_data, y):
        X_tr,  X_val  = X_data.iloc[tr_idx], X_data.iloc[val_idx]
        y_tr,  y_val  = y.iloc[tr_idx],      y.iloc[val_idx]
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_val)[:, 1]
        preds = (proba >= 0.5).astype(int)
        roc_scores.append(roc_auc_score(y_val, proba))
        pr_scores.append(average_precision_score(y_val, proba))
        f1_scores.append(f1_score(y_val, preds, average='macro'))
    print(f"\n── {name} ──")
    print(f"ROC-AUC:  {np.mean(roc_scores):.4f} ± {np.std(roc_scores):.4f}")
    print(f"PR-AUC:   {np.mean(pr_scores):.4f} ± {np.std(pr_scores):.4f}")
    print(f"F1-Macro: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
    return {
        'roc_auc':  np.mean(roc_scores),
        'pr_auc':   np.mean(pr_scores),
        'f1_macro': np.mean(f1_scores)
    }


rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=20,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.05,
    reg_lambda=0.1,
    scale_pos_weight=spw,
    eval_metric='aucpr',
    random_state=42,
    verbosity=0,
    n_jobs=-1
)

print("Running cross-validation...\n")
rf_scores  = run_cv(rf_model,  X_encoded, "Random Forest")
xgb_scores = run_cv(xgb_model, X_encoded, "XGBoost")

Running cross-validation...


── Random Forest ──
ROC-AUC:  0.7998 ± 0.0053
PR-AUC:   0.2752 ± 0.0103
F1-Macro: 0.5953 ± 0.0036

── XGBoost ──
ROC-AUC:  0.8209 ± 0.0038
PR-AUC:   0.3104 ± 0.0070
F1-Macro: 0.6114 ± 0.0050


In [16]:
all_scores = {
    'LightGBM':     lgbm_scores,
    'XGBoost':      xgb_scores,
    'Random Forest': rf_scores,
}

print("\n── Model Comparison ──")
print(f"{'Model':<20} {'ROC-AUC':>10} {'PR-AUC':>10} {'F1-Macro':>10}")
print("─" * 52)
for name, scores in all_scores.items():
    print(f"{name:<20} {scores['roc_auc']:>10.4f} "
          f"{scores['pr_auc']:>10.4f} {scores['f1_macro']:>10.4f}")


── Model Comparison ──
Model                   ROC-AUC     PR-AUC   F1-Macro
────────────────────────────────────────────────────
LightGBM                 0.8052     0.2839     0.6198
XGBoost                  0.8209     0.3104     0.6114
Random Forest            0.7998     0.2752     0.5953
